# RAG de Multas v2 — Mejorado

Mejoras sobre v1:
1. **Modelo multilingue** mejora para PDFs en espanol
2. **Cosine distance** metrica correcta para NLP
3. **Hybrid search** dense semantico + sparse BM25 keywords
4. **LLM conectado** Ollama local, gratis, sin API key

In [1]:
# Si te falta algo, descomenta y ejecuta:
# !pip install pypdf sentence-transformers chromadb rank-bm25 ollama
#
# Ollama debe estar corriendo en tu maquina:
# 1. Descarga Ollama: https://ollama.com/download
# 2. En terminal ejecuta: ollama pull llama3.2:1b
# 3. Ollama corre automaticamente en http://localhost:11434

In [12]:
import os
import torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login

In [ ]:
DOCUMENTS_PATH = './documents'
DB_PATH = './chromadb'
COLLECTION_NAME = 'Multas'
EMBEDDING_MODEL = os.getenv('EMBEDDING_MODEL', 'paraphrase-multilingual-MiniLM-L12-v2')
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
ALPHA = 0.7
GEMMA_MODEL = 'google/gemma-2b'
FALLBACK_MODEL = 'sshleifer/tiny-gpt2'
# Importante: NO hardcodear tokens en el notebook
HUGGINGFACE_TOKEN = os.getenv('HUGGINGFACE_HUB_TOKEN', '')  # Poner en variables de entorno

In [ ]:
# Configurar el token de Hugging Face para descargar Gemma directamente si está presente
if HUGGINGFACE_TOKEN:
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HUGGINGFACE_TOKEN
    try:
        login(token=HUGGINGFACE_TOKEN)
    except Exception as e:
        print('Warning: login failed', e)
else:
    print('HUGGINGFACE_HUB_TOKEN no encontrado en entorno; continuando sin login')

# Si no tienes GPU, Gemma-2b puede ser lento en CPU. Considera usar una versión más pequeña o GPU.

In [4]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk)
        start += (chunk_size - overlap)
    return chunks


def read_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ''
    for page in reader.pages:
        extracted = page.extract_text() or ''
        text += extracted + '\n'
    return text

In [5]:
# FIX 2: cosine distance en vez de L2 (euclidean por defecto de ChromaDB)
client = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={'hnsw:space': 'cosine'}
)
model = SentenceTransformer(EMBEDDING_MODEL)

bm25_corpus = []
bm25_index = None

print('Modelo:', EMBEDDING_MODEL)
print('Coleccion:', COLLECTION_NAME)
print('Distancia: cosine')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1132.02it/s]


Modelo: paraphrase-multilingual-MiniLM-L12-v2
Coleccion: Multas
Distancia: cosine


In [6]:
def ingest_documents(reset=False):
    global collection, bm25_corpus, bm25_index

    if reset:
        try:
            client.delete_collection(COLLECTION_NAME)
        except Exception:
            pass
        collection = client.get_or_create_collection(
            name=COLLECTION_NAME,
            metadata={'hnsw:space': 'cosine'}
        )
        bm25_corpus = []

    if not os.path.isdir(DOCUMENTS_PATH):
        raise FileNotFoundError(f'No existe la carpeta: {DOCUMENTS_PATH}')

    files = [f for f in os.listdir(DOCUMENTS_PATH) if f.lower().endswith('.pdf')]
    if not files:
        raise FileNotFoundError('No hay PDFs en ./documents')

    total = 0
    for file in files:
        file_path = os.path.join(DOCUMENTS_PATH, file)
        text = read_pdf(file_path)
        chunks = chunk_text(text)

        for idx, chunk in enumerate(chunks):
            chunk_id = f'{file}_{idx}'
            existing = collection.get(ids=[chunk_id])
            if not existing['ids']:
                emb = model.encode(chunk).tolist()
                collection.add(
                    documents=[chunk],
                    embeddings=[emb],
                    ids=[chunk_id],
                    metadatas=[{'source': file, 'chunk': idx}]
                )
            bm25_corpus.append(chunk)
            total += 1

    tokenized = [c.lower().split() for c in bm25_corpus]
    bm25_index = BM25Okapi(tokenized)

    return {'pdfs': len(files), 'chunks': total}

In [7]:
result = ingest_documents(reset=False)
result

invalid pdf header: b'\t\n%PD'
EOF marker seems truncated
incorrect startxref pointer(1)
parsing for Object Streams


{'pdfs': 8, 'chunks': 2777}

In [8]:
# FIX 3: Hybrid search
def hybrid_search(query, k=3, alpha=ALPHA):
    q_emb = model.encode(query).tolist()
    dense_res = collection.query(
        query_embeddings=[q_emb],
        n_results=k * 2
    )
    dense_docs  = dense_res.get('documents',  [[]])[0]
    dense_metas = dense_res.get('metadatas',  [[]])[0]
    dense_dists = dense_res.get('distances',  [[]])[0]

    tokenized_query = query.lower().split()
    bm25_scores = bm25_index.get_scores(tokenized_query)
    max_bm25 = max(bm25_scores) if max(bm25_scores) > 0 else 1
    bm25_norm = [s / max_bm25 for s in bm25_scores]
    bm25_top_idx = sorted(range(len(bm25_norm)), key=lambda i: bm25_norm[i], reverse=True)[:k * 2]

    scores = {}

    for doc, meta, dist in zip(dense_docs, dense_metas, dense_dists):
        key = meta.get('source', '') + str(meta.get('chunk', ''))
        dense_score = (1 - dist) * alpha
        scores[key] = scores.get(key, {'doc': doc, 'meta': meta, 'score': 0})
        scores[key]['score'] += dense_score

    for idx in bm25_top_idx:
        chunk = bm25_corpus[idx]
        sparse_score = bm25_norm[idx] * (1 - alpha)
        key = f'bm25_{idx}'
        if key not in scores:
            scores[key] = {'doc': chunk, 'meta': {'source': 'bm25', 'chunk': idx}, 'score': 0}
        scores[key]['score'] += sparse_score

    ranked = sorted(scores.values(), key=lambda x: x['score'], reverse=True)[:k]

    out = []
    for i, item in enumerate(ranked):
        doc = item['doc']
        out.append({
            'rank': i + 1,
            'source': item['meta'].get('source'),
            'score': round(item['score'], 4),
            'snippet': doc[:350] + ('...' if len(doc) > 350 else '')
        })
    return out

In [18]:
# FIX 4: LLM con Gemma — Hugging Face token

def ask_agent(query, k=3, alpha=ALPHA):
    chunks = hybrid_search(query, k=k, alpha=alpha)

    if not chunks:
        return {'error': 'No se encontro contexto relevante en los documentos.'}

    context_parts = []
    for c in chunks:
        context_parts.append('[Fuente: ' + c['source'] + ' | Score: ' + str(c['score']) + ']\n' + c['snippet'])
    context = '\n\n---\n\n'.join(context_parts)

    system_prompt = (
        'Eres un agente experto en multas de transito en Colombia. '
        'Tu funcion es ayudar a los ciudadanos a entender sus fotomultas y el proceso de apelacion. '
        'Basandote UNICAMENTE en el contexto legal proporcionado, responde de forma clara y estructurada. '
        'Si el contexto no es suficiente para responder con certeza, indicalo explicitamente. '
        'Nunca inventes articulos, leyes o procedimientos que no esten en el contexto.'
    )

    user_prompt = (
        'CONTEXTO LEGAL RECUPERADO:\n' + context + '\n\n'
        'CONSULTA DEL CIUDADANO:\n' + query + '\n\n'
        'Por favor responde indicando:\n'
        '1. Se puede apelar esta multa? Por que?\n'
        '2. Como es el proceso de apelacion paso a paso?\n'
        '3. Cual es la probabilidad estimada de exito? (Alta / Media / Baja) y por que.\n'
        '4. Hay algun plazo o requisito critico a tener en cuenta?'
    )

    full_prompt = f"{system_prompt}\n\n{user_prompt}"
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    try:
        tokenizer = AutoTokenizer.from_pretrained(
            GEMMA_MODEL,
            use_auth_token=HUGGINGFACE_TOKEN,
            trust_remote_code=True
        )
        model = AutoModelForCausalLM.from_pretrained(
            GEMMA_MODEL,
            use_auth_token=HUGGINGFACE_TOKEN,
            trust_remote_code=True
        )
    except OSError:
        print('No se tiene acceso al repositorio GEMMA. Usando modelo fallback:', FALLBACK_MODEL)
        tokenizer = AutoTokenizer.from_pretrained(FALLBACK_MODEL)
        model = AutoModelForCausalLM.from_pretrained(FALLBACK_MODEL)

    model = model.to(device)
    inputs = tokenizer(full_prompt, return_tensors='pt').to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )
    response_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return {
        'respuesta': response_text.strip(),
        'chunks_usados': chunks
    }

In [ ]:
# Prueba hybrid search solo
hybrid_search('Como impugnar una fotomulta en Colombia?', k=3)

[{'rank': 1,
  'source': 'bm25',
  'score': np.float64(0.3),
  'snippet': 'imiento regulado en la ley, se ajuste a las reglas básicas derivadas\ndel artículo 29 de la Constitución, como son, la existencia de un proceso\npúblico  sin  dilaciones  injustificadas,  la  oportunidad  de  controvertir  e\nimpugnar las decisiones, la garantía del derecho de defensa y la posibilidad\nde presentar y controvertir pruebas, con lo cual s...'},
 {'rank': 2,
  'source': 'bm25',
  'score': np.float64(0.3),
  'snippet': 'tales, y la plena observancia de las demás disposiciones\nconstitucionales. \nEn relación con esto último, se debe destacar que el derecho al debido proceso\nexige que todo procedimiento regulado en la ley, se ajuste a las reglas básicas\nderivadas del artículo 29 de la Constitución, como son, la existencia de un\nproceso público sin dilaciones injustif...'},
 {'rank': 3,
  'source': 'bm25',
  'score': np.float64(0.2955),
  'snippet': 'siguientes a la notiﬁcación del comparendo, la au

In [21]:
# Prueba pipeline completo con Gemma/fallback
respuesta = ask_agent('Me pusieron una fotomulta por exceso de velocidad, puedo apelarla?')
print(respuesta['respuesta'][:1000])
if len(respuesta['respuesta']) > 1000:
    print('\n...respuesta truncada...')
print('\n--- Chunks usados ---')
for c in respuesta['chunks_usados']:
    print(f"  [{c['rank']}] {c['source']} | score: {c['score']} | snippet: {c['snippet'][:120].replace('\n',' ')}...")

No se tiene acceso al repositorio GEMMA. Usando modelo fallback: sshleifer/tiny-gpt2


Loading weights: 100%|██████████| 29/29 [00:00<00:00, 5762.77it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Eres un agente experto en multas de transito en Colombia. Tu funcion es ayudar a los ciudadanos a entender sus fotomultas y el proceso de apelacion. Basandote UNICAMENTE en el contexto legal proporcionado, responde de forma clara y estructurada. Si el contexto no es suficiente para responder con certeza, indicalo explicitamente. Nunca inventes articulos, leyes o procedimientos que no esten en el contexto.

CONTEXTO LEGAL RECUPERADO:
[Fuente: Ley_769_de_2002.pdf | Score: 0.3671]
velocidades entre treinta (30) y sesenta (60) kilómetros por hora, veinte (20) metros.
Para velocidades entre sesenta (60) y ochenta (80) kilómetros por hora, veinticinco (25) metros.
Para velocidades de ochenta (80) kilómetros en adelante, treinta (30) metros o la que la autoridad competente indique.
En todos los casos, el conductor deberá atender...

---

[Fuente: Ley_769_de_2002.pdf | Score: 0.314]
o por el art. 2, Decreto Nacional 15 de 2011
ARTÍCULO 108. SEPARACIÓN ENTRE VEHÍCULOS. La separación entre dos (